In [1]:
import duckdb

con = duckdb.connect()

transaction_file = r"C:\kavya\redischool\fraud detection\data\transactions_data.csv"

# Inspect column names and data types
schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto('{transaction_file}', sample_size=100000)
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,date,TIMESTAMP,YES,None,None,None
2,client_id,BIGINT,YES,None,None,None
3,card_id,BIGINT,YES,None,None,None
4,amount,VARCHAR,YES,None,None,None
5,use_chip,VARCHAR,YES,None,None,None
6,merchant_id,BIGINT,YES,None,None,None
7,merchant_city,VARCHAR,YES,None,None,None
8,merchant_state,VARCHAR,YES,None,None,None
9,zip,DOUBLE,YES,None,None,None


In [4]:
cards_file = r"C:\kavya\redischool\fraud detection\data\cards_data.csv"

card_summary = con.execute(f"""
    SELECT
        card_type,
        COUNT(*) AS total_cards
    FROM read_csv_auto('{cards_file}')
    GROUP BY card_type
    ORDER BY total_cards DESC
""").df()

display(card_summary)

,card_type,total_cards
0,Debit,3511
1,Credit,2057
2,Debit (Prepaid),578


In [6]:
transaction_methods = con.execute(f"""
    SELECT
        use_chip,
        COUNT(*) AS total_transactions
    FROM read_csv(
        '{transaction_file}',
        header = true,
        auto_detect = true,
        strict_mode = false,
        null_padding = true
    )
    GROUP BY use_chip
    ORDER BY total_transactions DESC
""").df()

display(transaction_methods)

,use_chip,total_transactions
0,Swipe Transaction,6967185
1,Chip Transaction,4780818
2,Online Transaction,1557912


In [7]:
debit_summary = con.execute(f"""
    SELECT
        c.card_type,
        COUNT(*) AS total_transactions
    FROM read_csv(
        '{transaction_file}',
        header = true,
        quote = '"',
        escape = '"',
        strict_mode = false,
        null_padding = true
    ) t
    JOIN read_csv_auto('{cards_file}') c
        ON t.card_id = c.id
    GROUP BY c.card_type
    ORDER BY total_transactions DESC
""").df()

display(debit_summary)

,card_type,total_transactions
0,Debit,8280996
1,Credit,4109189
2,Debit (Prepaid),915730


In [8]:
{
    "target": {
        "10649266": "No",
        "23410063": "No"
    }
}

{'target': {'10649266': 'No', '23410063': 'No'}}

In [9]:
import json
from collections import Counter

labels_file = r"C:\kavya\redischool\fraud detection\data\train_fraud_labels.json"

with open(labels_file, "r") as f:
    fraud_data = json.load(f)

labels = fraud_data["target"]

print("Total labelled transactions:", len(labels))
print("\nFraud label distribution:")
print(Counter(labels.values()))

Total labelled transactions: 8914963

Fraud label distribution:
Counter({'No': 8901631, 'Yes': 13332})


In [13]:
import os

folder = r"C:\kavya\redischool\fraud detection\data"

print("Folder exists:", os.path.exists(folder))
print("Folder writable:", os.access(folder, os.W_OK))

test_file = os.path.join(folder, "test.txt")

with open(test_file, "w") as f:
    f.write("Testing file access")

print("File created successfully")

import csv

labels_csv = os.path.join(folder, "fraud_labels.csv")

with open(labels_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["transaction_id", "fraud_label"])
    writer.writerows(labels.items())

print("Labels exported successfully:", labels_csv)

Folder exists: True
Folder writable: True
File created successfully
Labels exported successfully: C:\kavya\redischool\fraud detection\data\fraud_labels.csv


In [14]:
labels_csv = r"C:\kavya\redischool\fraud detection\data\fraud_labels.csv"

fraud_by_card = con.execute(f"""
    SELECT
        c.card_type,
        COUNT(*) AS labelled_transactions,
        SUM(CASE WHEN l.fraud_label = 'Yes' THEN 1 ELSE 0 END)
            AS fraud_transactions,
        SUM(CASE WHEN l.fraud_label = 'No' THEN 1 ELSE 0 END)
            AS genuine_transactions,
        ROUND(
            100.0 * SUM(CASE WHEN l.fraud_label = 'Yes' THEN 1 ELSE 0 END)
            / COUNT(*), 4
        ) AS fraud_percentage
    FROM read_csv(
        '{transaction_file}',
        header = true,
        quote = '"',
        escape = '"',
        strict_mode = false,
        null_padding = true
    ) t
    JOIN read_csv_auto('{cards_file}') c
        ON t.card_id = c.id
    JOIN read_csv_auto('{labels_csv}') l
        ON CAST(t.id AS VARCHAR) = l.transaction_id
    GROUP BY c.card_type
    ORDER BY c.card_type
""").df()

display(fraud_by_card)

,card_type,labelled_transactions,fraud_transactions,genuine_transactions,fraud_percentage
0,Credit,2753467,4490.0,2748977.0,0.1631
1,Debit,5548196,7464.0,5540732.0,0.1345
2,Debit (Prepaid),613300,1378.0,611922.0,0.2247


In [15]:
debit_output = r"C:\kavya\redischool\fraud detection\data\debit_fraud_dataset.csv"

con.execute(f"""
    COPY (
        SELECT
            t.id AS transaction_id,
            t.date,
            t.client_id,
            t.card_id,
            TRY_CAST(REPLACE(t.amount, '$', '') AS DOUBLE) AS amount,
            t.use_chip,
            t.merchant_id,
            t.merchant_city,
            t.merchant_state,
            t.mcc,
            t.errors,
            l.fraud_label
        FROM read_csv(
            '{transaction_file}',
            header = true,
            quote = '"',
            escape = '"',
            strict_mode = false,
            null_padding = true
        ) t
        JOIN read_csv_auto('{cards_file}') c
            ON t.card_id = c.id
        JOIN read_csv_auto('{labels_csv}') l
            ON CAST(t.id AS VARCHAR) = l.transaction_id
        WHERE c.card_type = 'Debit'
    )
    TO '{debit_output}'
    (HEADER, DELIMITER ',')
""")

print("Debit dataset exported successfully!")

Debit dataset exported successfully!


In [16]:
debit_file = r"C:\kavya\redischool\fraud detection\data\debit_fraud_dataset.csv"

debit_validation = con.execute(f"""
    SELECT
        COUNT(*) AS total_transactions,
        COUNT(DISTINCT transaction_id) AS unique_transactions,
        COUNT(*) FILTER (WHERE fraud_label = 'Yes') AS fraud_transactions,
        COUNT(*) FILTER (WHERE fraud_label = 'No') AS genuine_transactions,
        COUNT(*) FILTER (WHERE amount IS NULL) AS missing_amounts
    FROM read_csv_auto('{debit_file}')
""").df()

display(debit_validation)

fraud_distribution = con.execute(f"""
    SELECT
        fraud_label,
        COUNT(*) AS transaction_count,
        ROUND(100.0 * COUNT(*) /
            SUM(COUNT(*)) OVER (), 4) AS percentage
    FROM read_csv_auto('{debit_file}')
    GROUP BY fraud_label
""").df()

display(fraud_distribution)

,total_transactions,unique_transactions,fraud_transactions,genuine_transactions,missing_amounts
0,5548196,5548196,7464,5540732,0


,fraud_label,transaction_count,percentage
0,False,5540732,99.8655
1,True,7464,0.1345


In [17]:
import pandas as pd

df = pd.read_csv(debit_file)

print("Dataset shape:", df.shape)

print("\nColumn information:")
df.info()

print("\nFirst 5 records:")
display(df.head())

Dataset shape: (5548196, 12)

Column information:
<class 'pandas.DataFrame'>
RangeIndex: 5548196 entries, 0 to 5548195
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   transaction_id  int64  
 1   date            str    
 2   client_id       int64  
 3   card_id         int64  
 4   amount          float64
 5   use_chip        str    
 6   merchant_id     int64  
 7   merchant_city   str    
 8   merchant_state  str    
 9   mcc             int64  
 10  errors          str    
 11  fraud_label     bool   
dtypes: bool(1), float64(1), int64(5), str(5)
memory usage: 717.9 MB

First 5 records:


,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,mcc,errors,fraud_label
0,7678340,2010-02-21 13:53:00,1105,218,80.00,Swipe Transaction,27092,Leander,TX,4829,NaN,False
1,7678341,2010-02-21 13:53:00,1116,5559,6.53,Swipe Transaction,22879,Telford,PA,5411,NaN,False
2,7678344,2010-02-21 13:54:00,790,5123,75.09,Swipe Transaction,62087,Lakeside,CA,5300,NaN,False
3,7678350,2010-02-21 13:57:00,135,2808,39.32,Online Transaction,39021,ONLINE,NaN,4784,NaN,False
4,7678357,2010-02-21 13:58:00,1856,257,-76.00,Swipe Transaction,43293,Fayetteville,NC,5499,NaN,False


In [18]:
# Check missing values in each column

missing_values = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": (missing_values / len(df)) * 100
})

display(missing_summary)

,Missing_Count,Missing_Percentage
transaction_id,0,0.000000
date,0,0.000000
client_id,0,0.000000
card_id,0,0.000000
amount,0,0.000000
use_chip,0,0.000000
merchant_id,0,0.000000
merchant_city,0,0.000000
merchant_state,652450,11.759678
mcc,0,0.000000


In [19]:
fraud_by_method = df.groupby("use_chip").agg(
    total_transactions=("fraud_label", "count"),
    fraud_transactions=("fraud_label", "sum")
)

fraud_by_method["genuine_transactions"] = (
    fraud_by_method["total_transactions"]
    - fraud_by_method["fraud_transactions"]
)

fraud_by_method["fraud_percentage"] = (
    fraud_by_method["fraud_transactions"]
    / fraud_by_method["total_transactions"] * 100
)

display(fraud_by_method)

,total_transactions,fraud_transactions,genuine_transactions,fraud_percentage
use_chip,,,,
Chip Transaction,2001708,1660,2000048,0.082929
Online Transaction,650441,4986,645455,0.766557
Swipe Transaction,2896047,818,2895229,0.028245


In [20]:
df["has_error"] = df["errors"].notna()

fraud_by_error = df.groupby("has_error").agg(
    total_transactions=("fraud_label", "count"),
    fraud_transactions=("fraud_label", "sum")
)

fraud_by_error["fraud_percentage"] = (
    fraud_by_error["fraud_transactions"]
    / fraud_by_error["total_transactions"] * 100
)

display(fraud_by_error)

,total_transactions,fraud_transactions,fraud_percentage
has_error,,,
False,5456466,7129,0.130652
True,91730,335,0.365202


In [21]:
# Divide transactions into 10 groups based on transaction amount

df["amount_group"] = pd.qcut(
    df["amount"],
    q=10,
    duplicates="drop"
)

fraud_by_amount = df.groupby(
    "amount_group",
    observed=True
).agg(
    total_transactions=("fraud_label", "count"),
    fraud_transactions=("fraud_label", "sum")
)

fraud_by_amount["fraud_percentage"] = (
    fraud_by_amount["fraud_transactions"]
    / fraud_by_amount["total_transactions"] * 100
)

display(fraud_by_amount)

,total_transactions,fraud_transactions,fraud_percentage
amount_group,,,
"(-500.001, 1.61]",555307,627,0.112911
"(1.61, 5.15]",554761,423,0.076249
"(5.15, 10.9]",554406,434,0.078282
"(10.9, 17.38]",554878,320,0.057670
"(17.38, 26.82]",555159,358,0.064486
"(26.82, 38.18]",554500,413,0.074482
"(38.18, 52.13]",554879,511,0.092092
"(52.13, 70.76]",554808,664,0.119681
"(70.76, 100.0]",601030,860,0.143088


In [22]:
negative_amounts = df[df["amount"] < 0]

print("Negative amount transactions:", len(negative_amounts))
print("\nFraud distribution:")
display(negative_amounts["fraud_label"].value_counts())

print("\nTransaction methods:")
display(negative_amounts["use_chip"].value_counts())

Negative amount transactions: 274842

Fraud distribution:


fraud_label
False    274574
True        268
Name: count, dtype: int64


Transaction methods:


use_chip
Swipe Transaction     161773
Chip Transaction      110751
Online Transaction      2318
Name: count, dtype: int64

In [23]:
fraud_by_mcc = df.groupby("mcc").agg(
    total_transactions=("fraud_label", "count"),
    fraud_transactions=("fraud_label", "sum")
)

fraud_by_mcc["fraud_percentage"] = (
    fraud_by_mcc["fraud_transactions"]
    / fraud_by_mcc["total_transactions"] * 100
)

# Focus on categories with at least 1,000 transactions
fraud_by_mcc = fraud_by_mcc[
    fraud_by_mcc["total_transactions"] >= 1000
]

fraud_by_mcc = fraud_by_mcc.sort_values(
    "fraud_percentage", ascending=False
)

display(fraud_by_mcc.head(15))

,total_transactions,fraud_transactions,fraud_percentage
mcc,,,
5045,1031,116,11.251212
5732,2575,228,8.854369
5712,1321,98,7.418622
5816,1005,69,6.865672
5094,2021,119,5.888174
5932,2246,42,1.869991
5977,2705,50,1.848429
7996,4041,66,1.633259
3389,3786,61,1.611199


In [24]:
import json

mcc_file = r"C:\kavya\redischool\fraud detection\data\mcc_codes.json"

with open(mcc_file, "r") as f:
    mcc_codes = json.load(f)

fraud_by_mcc["merchant_category"] = (
    fraud_by_mcc.index.astype(str).map(mcc_codes)
)

display(fraud_by_mcc.head(15))

,total_transactions,fraud_transactions,fraud_percentage,merchant_category
mcc,,,,
5045,1031,116,11.251212,"Computers, Computer Peripheral Equipment"
5732,2575,228,8.854369,Electronics Stores
5712,1321,98,7.418622,"Furniture, Home Furnishings, and Equipment Stores"
5816,1005,69,6.865672,Digital Goods - Games
5094,2021,119,5.888174,Precious Stones and Metals
5932,2246,42,1.869991,Antique Shops
5977,2705,50,1.848429,Cosmetic Stores
7996,4041,66,1.633259,"Amusement Parks, Carnivals, Circuses"
3389,3786,61,1.611199,Non-Precious Metal Services


In [25]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Extract transaction hour
df["transaction_hour"] = df["date"].dt.hour

fraud_by_hour = df.groupby("transaction_hour").agg(
    total_transactions=("fraud_label", "count"),
    fraud_transactions=("fraud_label", "sum")
)

fraud_by_hour["fraud_percentage"] = (
    fraud_by_hour["fraud_transactions"]
    / fraud_by_hour["total_transactions"] * 100
)

display(fraud_by_hour)

,total_transactions,fraud_transactions,fraud_percentage
transaction_hour,,,
0,61898,9,0.014540
1,43355,22,0.050744
2,44573,21,0.047114
3,40295,62,0.153865
4,46823,78,0.166585
5,80710,146,0.180895
6,327351,250,0.076371
7,372309,305,0.081921
8,370509,299,0.080700
